# 상권정보 데이터 수집

In [1]:
import requests
import pandas as pd
import time

In [2]:
def unpack_data(response):
    result = {}
    for item in response['body']['items']:
        for key, value in zip(response['header']['columns'], item.values()):
            result.setdefault(key, []).append(value)
    df = pd.DataFrame(result)
    return df

In [3]:
page = 1
result_dfs = []
while True:
    url = "http://apis.data.go.kr/B553077/api/open/sdsc2/storeListInUpjong"
    service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
    payload = dict(servicekey=service_key, divId="indsSclsCd", 
                  key="I21006", numOfRows=1000, pageNo=page, type="json")
    r = requests.get(url, params=payload)
    print(r.url, end="\r")
    print(r.status_code, end="\r")
    
    response = r.json()
    total_page = (response['body']['totalCount'] // 1000) + 1
    result_dfs.append(unpack_data(response))
    print(f"{page}/{total_page} 수집중", end="\r")
    if page < total_page:
        page += 1
    else:
        break
    time.sleep(0.2)
        
result = pd.concat(result_dfs)    
result = result.reset_index(drop=True)
result.to_csv("공공데이터_상권정보_202504.csv", encoding="utf-8", index=False)

In [4]:
response

{'header': {'description': '소상공인시장진흥공단 주요상권내 상가업소정보',
  'columns': ['상가업소번호',
   '상호명',
   '지점명',
   '상권업종대분류코드',
   '상권업종대분류명',
   '상권업종중분류코드',
   '상권업종중분류명',
   '상권업종소분류코드',
   '상권업종소분류명',
   '표준산업분류코드',
   '표준산업분류명',
   '시도코드',
   '시도명',
   '시군구코드',
   '시군구명',
   '행정동코드',
   '행정동명',
   '법정동코드',
   '법정동명',
   'PNU코드',
   '대지구분코드',
   '대지구분명',
   '지번본번지',
   '지번부번지',
   '지번주소',
   '도로명코드',
   '도로명',
   '건물본번지',
   '건물부번지',
   '건물관리번호',
   '건물명',
   '도로명주소',
   '구우편번호',
   '신우편번호',
   '동정보',
   '층정보',
   '호정보',
   '경도',
   '위도'],
  'stdrYm': '202506',
  'resultCode': '00',
  'resultMsg': 'NORMAL SERVICE'},
 'body': {'items': [{'bizesId': 'MA0101202504A0107778',
    'bizesNm': '보드람치킨안양',
    'brchNm': '비산점',
    'indsLclsCd': 'I2',
    'indsLclsNm': '음식',
    'indsMclsCd': 'I210',
    'indsMclsNm': '기타 간이',
    'indsSclsCd': 'I21006',
    'indsSclsNm': '치킨',
    'ksicCd': 'I56193',
    'ksicNm': '치킨 전문점',
    'ctprvnCd': '41',
    'ctprvnNm': '경기도',
    'signguCd': '41173',
    'signguN

In [8]:
# !pip install openpyxl

In [6]:
ori_data = pd.read_excel("./data/소상공인시장진흥공단_상가(상권)정보_업종분류(2302)_및_연계표_v1.0.xlsx", engine='openpyxl', header=[1])
ori_data

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [15]:
ori_data['소분류코드']

0      G20201
1      G20202
2      G20301
3      G20404
4      G20405
        ...  
242    S20902
243    S21001
244    S21002
245    S21101
246    S21105
Name: 소분류코드, Length: 247, dtype: object

In [10]:
import os

In [20]:
errors = []
for idx, (code, name) in enumerate(zip(ori_data['소분류코드'][80:], ori_data['소분류명'][80:])):
    name = name.replace("/", "_").replace("·", "_")
    print(f"{name}_수집중 {idx}/{len(ori_data['소분류코드'])}", end="\r")
    page = 1
    result_dfs = []
    while True:
        url = "http://apis.data.go.kr/B553077/api/open/sdsc2/storeListInUpjong"
        service_key = "8Ym5dhmVJdr12XzGnyYrQjSBS1QuRBYK8yHfx65JCl4vACM9uLKo8fxCVOFJkPB71llD7F2rOEROZnHgNGoj3A=="
        payload = dict(servicekey=service_key, divId="indsSclsCd", 
                      key=code, numOfRows=1000, pageNo=page, type="json")
        r = requests.get(url, params=payload)
        print(r.url, end="\r")
        print(r.status_code, end="\r")

        response = r.json()
        try:
            total_page = (response['body']['totalCount'] // 1000) + 1
        except Exception as e:
            print("데이터 없음", end="\r")
            errors.append(f"{idx}, {code}, {name} 데이터 없음")
            break
        result_dfs.append(unpack_data(response))
        print(f"{page}/{total_page} 수집중", end="\r")
        if page < total_page:
            page += 1
        else:
            break
        time.sleep(0.2)

    result = pd.concat(result_dfs)    
    result = result.reset_index(drop=True)
    
    # 폴더가 있는지 확인하고 없으면 폴더 생성하기
    if not os.path.exists("./scraping_results"):
        os.mkdir("./scraping_results")
#     else:
#         print("scraping_results이 이미 있습니다.")
    
    result.to_csv(f"./scraping_results/공공데이터_{name}_상권정보_202504.csv", encoding="utf-8", index=False)

In [18]:
name

'횟집'

In [19]:
response

{'header': {'description': '소상공인시장진흥공단 주요상권내 상가업소정보',
  'columns': ['상가업소번호',
   '상호명',
   '지점명',
   '상권업종대분류코드',
   '상권업종대분류명',
   '상권업종중분류코드',
   '상권업종중분류명',
   '상권업종소분류코드',
   '상권업종소분류명',
   '표준산업분류코드',
   '표준산업분류명',
   '시도코드',
   '시도명',
   '시군구코드',
   '시군구명',
   '행정동코드',
   '행정동명',
   '법정동코드',
   '법정동명',
   'PNU코드',
   '대지구분코드',
   '대지구분명',
   '지번본번지',
   '지번부번지',
   '지번주소',
   '도로명코드',
   '도로명',
   '건물본번지',
   '건물부번지',
   '건물관리번호',
   '건물명',
   '도로명주소',
   '구우편번호',
   '신우편번호',
   '동정보',
   '층정보',
   '호정보',
   '경도',
   '위도'],
  'resultCode': '03',
  'resultMsg': 'NODATA_ERROR'},
 'body': {'items': []}}